# Build a RAG Pipeline from Scratch in Python

> **Chat with your PDF** using OpenAI + ChromaDB — no frameworks, no magic, just ~50 lines of core logic.

## What You'll Learn

1. **Load & chunk** a PDF document
2. **Generate embeddings** using OpenAI's embedding model
3. **Store vectors** in ChromaDB (open-source vector database)
4. **Retrieve** relevant chunks for a user query
5. **Generate** a grounded answer using GPT — with source citations

## Architecture

```
PDF → Chunks → Embeddings → ChromaDB (Vector Store)
                                    ↓
User Query → Embed Query → Similarity Search → Top-K Chunks
                                                      ↓
                                    LLM Prompt (Query + Context) → Answer
```

## Prerequisites

- Python 3.9+
- An [OpenAI API key](https://platform.openai.com/api-keys)

---

**Part of [Awesome GenAI Toolkit](https://github.com/shubh-vedi/awesome-genai-toolkit)** — Star the repo if this helps you!

## Step 0: Install Dependencies

In [ ]:
!pip install -q openai chromadb pymupdf

## Step 1: Setup & Configuration

In [ ]:
import os
import fitz  # PyMuPDF
import chromadb
from openai import OpenAI

# --- Configuration ---
# Option 1: Set your API key here (for quick testing)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Option 2: Use Google Colab secrets (recommended)
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass  # Not running in Colab

client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"  # $0.02 / 1M tokens — very cheap
CHAT_MODEL = "gpt-4o-mini"                  # $0.15 / 1M input tokens
CHUNK_SIZE = 500                             # characters per chunk
CHUNK_OVERLAP = 50                           # overlap between chunks

print("Setup complete!")

## Step 2: Load a PDF and Extract Text

We use **PyMuPDF** (`fitz`) — it's fast, lightweight, and handles most PDFs well.

You can upload your own PDF or use the sample below.

In [ ]:
# --- Upload or use a sample PDF ---

# For Google Colab: upload a file
# from google.colab import files
# uploaded = files.upload()
# PDF_PATH = list(uploaded.keys())[0]

# For local: set the path to your PDF
PDF_PATH = "sample.pdf"  # <-- Change this to your PDF path

# --- Create a sample PDF for demo purposes ---
def create_sample_pdf(path: str):
    """Create a sample PDF about AI agents for testing."""
    sample_text = """
Chapter 1: Introduction to AI Agents

An AI agent is an autonomous system that perceives its environment, makes decisions, and takes
actions to achieve specific goals. Unlike traditional software that follows rigid rules, AI agents
use large language models (LLMs) to reason, plan, and adapt to new situations.

The core components of an AI agent are:
1. Perception: The ability to understand inputs (text, images, audio)
2. Reasoning: Using an LLM to analyze information and plan next steps
3. Action: Executing tools, APIs, or code to accomplish tasks
4. Memory: Storing context from previous interactions for continuity

Chapter 2: Types of AI Agents

There are several architectures for building AI agents:

ReAct Agents: These agents alternate between reasoning (thinking) and acting (using tools).
The agent first thinks about what to do, then executes a tool, observes the result, and
repeats until the task is complete. This is the most common pattern used in frameworks
like LangGraph and CrewAI.

Plan-and-Execute Agents: These agents first create a complete plan, then execute each step
sequentially. They are better for complex, multi-step tasks where upfront planning reduces
errors.

Multi-Agent Systems: Multiple specialized agents collaborate on a task. For example, a
researcher agent gathers information, a writer agent drafts content, and an editor agent
reviews quality. Frameworks like CrewAI and AutoGen support this pattern.

Chapter 3: RAG — Retrieval Augmented Generation

RAG is a technique that enhances LLM responses by retrieving relevant information from external
knowledge bases before generating an answer. This approach solves several problems:

- Reduces hallucinations by grounding responses in real data
- Keeps answers up-to-date without retraining the model
- Allows LLMs to work with private or domain-specific data

A basic RAG pipeline works in three steps:
1. Indexing: Documents are split into chunks, embedded into vectors, and stored in a vector database
2. Retrieval: When a user asks a question, the query is embedded and similar chunks are retrieved
3. Generation: The retrieved chunks are passed as context to the LLM, which generates a grounded answer

Chapter 4: Advanced RAG Patterns

Agentic RAG combines AI agents with RAG to create self-correcting retrieval systems. The agent
can decide when to retrieve, what to retrieve, and whether the retrieved information is sufficient.
If the context is not good enough, the agent can reformulate the query and try again.

Hybrid Search combines vector similarity search with traditional keyword search (BM25) to
improve retrieval quality. Vector search is good at finding semantically similar content, while
keyword search excels at finding exact matches. Combining both gives the best results.

HyDE (Hypothetical Document Embeddings) is a technique where the LLM first generates a
hypothetical answer, then uses that answer's embedding to search for similar real documents.
This bridges the gap between question embeddings and document embeddings.

Chapter 5: Tools and Frameworks

Popular vector databases for RAG include ChromaDB, Pinecone, Weaviate, Milvus, and Qdrant.
ChromaDB is the easiest to start with — it runs locally, requires no setup, and is fully
open-source.

For building agents, LangGraph provides stateful workflows with cycles and persistence.
CrewAI enables multi-agent collaboration with role-based teams. AutoGen from Microsoft
supports conversable agents that can execute code.

The Model Context Protocol (MCP) is a new open standard that allows LLMs to securely connect
to external tools, databases, and APIs through a unified interface.
"""
    doc = fitz.open()
    for i in range(0, len(sample_text), 2000):
        page = doc.new_page()
        page.insert_text((72, 72), sample_text[i:i+2000], fontsize=11)
    doc.save(path)
    doc.close()
    print(f"Sample PDF created at: {path}")

# Create sample PDF if no file exists
if not os.path.exists(PDF_PATH):
    create_sample_pdf(PDF_PATH)
    print("(Using built-in sample PDF about AI Agents & RAG)")
else:
    print(f"Using existing PDF: {PDF_PATH}")

In [ ]:
def extract_text_from_pdf(path: str) -> str:
    """Extract all text from a PDF file."""
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

raw_text = extract_text_from_pdf(PDF_PATH)
print(f"Extracted {len(raw_text)} characters from {PDF_PATH}")
print(f"Preview: {raw_text[:300]}...")

## Step 3: Chunk the Text

LLMs have limited context windows, and embedding models work best on small, focused passages. We split the text into overlapping chunks.

**Why overlap?** So that sentences split across chunk boundaries aren't lost.

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        # Try to break at a sentence boundary
        if end < len(text):
            last_period = chunk.rfind(". ")
            last_newline = chunk.rfind("\n")
            break_point = max(last_period, last_newline)
            if break_point > chunk_size // 2:
                end = start + break_point + 1
                chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
    return [c for c in chunks if len(c) > 50]  # Filter tiny chunks

chunks = chunk_text(raw_text)
print(f"Created {len(chunks)} chunks")
print(f"\n--- Chunk 0 (preview) ---\n{chunks[0][:200]}...")
print(f"\n--- Chunk 1 (preview) ---\n{chunks[1][:200]}...")

## Step 4: Generate Embeddings & Store in ChromaDB

We embed each chunk into a vector and store it in ChromaDB. ChromaDB runs locally — no server setup needed.

In [ ]:
def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Get embeddings for a list of texts using OpenAI."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]

# --- Store in ChromaDB ---
chroma_client = chromadb.Client()  # In-memory for demo; use PersistentClient for production
collection = chroma_client.create_collection(name="rag_demo")

# Embed and store all chunks
print("Generating embeddings...")
embeddings = get_embeddings(chunks)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks,
    metadatas=[{"chunk_index": i, "source": PDF_PATH} for i in range(len(chunks))]
)

print(f"Stored {collection.count()} chunks in ChromaDB")

## Step 5: Retrieve Relevant Chunks

Given a user query, we embed it and find the most similar chunks using cosine similarity.

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[str]:
    """Retrieve the top-K most relevant chunks for a query."""
    query_embedding = get_embeddings([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results["documents"][0]

# Test retrieval
test_query = "What is RAG and how does it work?"
retrieved_chunks = retrieve(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved_chunks)} chunks:\n")
for i, chunk in enumerate(retrieved_chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk[:200])
    print()

## Step 6: Generate an Answer with Context (The RAG Part)

We pass the retrieved chunks as context to the LLM, instructing it to answer **only** from the provided context.

In [ ]:
RAG_PROMPT = """You are a helpful assistant that answers questions based on the provided context.

Rules:
- Answer ONLY from the context below. Do not use prior knowledge.
- If the context doesn't contain the answer, say "I don't have enough information to answer this."
- Cite which chunk(s) your answer comes from.
- Be concise and direct.

Context:
{context}

Question: {question}
"""

def rag_query(question: str, top_k: int = 3) -> str:
    """Full RAG pipeline: retrieve context, then generate answer."""
    # 1. Retrieve
    chunks = retrieve(question, top_k=top_k)
    context = "\n\n".join(
        f"[Chunk {i+1}]: {chunk}" for i, chunk in enumerate(chunks)
    )

    # 2. Generate
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": RAG_PROMPT.format(context=context, question=question)}
        ],
        temperature=0
    )
    return response.choices[0].message.content

# --- Try it! ---
answer = rag_query("What is RAG and how does it work?")
print(answer)

## Step 7: Interactive Chat with Your PDF

Let's make it conversational! Ask multiple questions about your document.

In [ ]:
# --- Try different questions ---
questions = [
    "What are the core components of an AI agent?",
    "What is the difference between ReAct agents and Plan-and-Execute agents?",
    "How does Agentic RAG differ from basic RAG?",
    "What is HyDE?",
    "Which vector databases are mentioned?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_query(q)}")
    print("-" * 80)

## Step 8: Evaluate Retrieval Quality

A simple way to check if your RAG pipeline is working: verify that the retrieved chunks actually contain the answer.

In [ ]:
def evaluate_retrieval(question: str, expected_keywords: list[str], top_k: int = 3):
    """Check if retrieved chunks contain expected keywords."""
    chunks = retrieve(question, top_k=top_k)
    combined = " ".join(chunks).lower()

    results = {}
    for keyword in expected_keywords:
        results[keyword] = keyword.lower() in combined

    found = sum(results.values())
    total = len(results)
    print(f"Query: {question}")
    print(f"Keyword hit rate: {found}/{total} ({found/total*100:.0f}%)")
    for kw, found in results.items():
        status = "FOUND" if found else "MISSING"
        print(f"  [{status}] {kw}")
    print()

# --- Run evaluation ---
evaluate_retrieval(
    "What is RAG?",
    ["retrieval", "augmented", "generation", "hallucination", "vector"]
)

evaluate_retrieval(
    "What types of AI agents exist?",
    ["ReAct", "plan-and-execute", "multi-agent", "CrewAI"]
)

evaluate_retrieval(
    "What is HyDE?",
    ["hypothetical", "document", "embedding"]
)

## Bonus: RAG with Your Own PDF

Replace `PDF_PATH` in Step 2 with your own file and re-run all cells. That's it — you now have a **Chat with PDF** app!

### Ideas to Try
- Upload a research paper and ask it questions
- Upload your company's documentation
- Upload a textbook chapter and use it as a study tool

## What's Next?

You've built a RAG pipeline from scratch. Here's how to level up:

| Improvement | Description | Notebook |
|-------------|-------------|----------|
| **Agentic RAG** | Self-correcting retrieval with LangGraph | Coming soon |
| **Hybrid Search** | Combine vector + keyword search (BM25) | Coming soon |
| **RAG Evaluation** | Measure with Ragas (faithfulness, relevancy) | Coming soon |
| **Multimodal RAG** | Search images + text + PDFs with Gemini | Coming soon |

---

### Found this useful?

**[Star the Awesome GenAI Toolkit repo](https://github.com/shubh-vedi/awesome-genai-toolkit)** — it helps others discover these tutorials!

More notebooks, apps, and guides are being added every week. Contributions welcome!